In [ ]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

In [ ]:
!pip install -q transformers accelerate bitsandbytes


In [ ]:
!pip install -q transformers accelerate bitsandbytes


In [ ]:
from transformers import AutoTokenizer, AutoModelForCausalLM, pipeline

model_id = "mistralai/Mistral-7B-Instruct-v0.2"

tokenizer = AutoTokenizer.from_pretrained(model_id)
model = AutoModelForCausalLM.from_pretrained(
    model_id,
    torch_dtype="auto",
    device_map="auto"
)

pipe = pipeline("text-generation", model=model, tokenizer=tokenizer)


In [ ]:
pipe = pipeline(
    "text-generation",
    model=model,
    tokenizer=tokenizer,
    torch_dtype="auto",
    device_map="auto"
)


In [ ]:
prompt = """You are a medical assistant.
Given cardiac EF data, write a short diagnostic summary.

Example:
EF: 65% → Normal systolic function.
EF: 45% → Mild systolic dysfunction.
EF: 30% → Severe systolic dysfunction.

Now respond for EF: 38%."""

result = pipe(prompt, max_new_tokens=80, do_sample=False)
print(result[0]["generated_text"])


In [ ]:
import pandas as pd
from transformers import pipeline

# ==========================================================
# Load EF data
# ==========================================================
df = pd.read_csv("/kaggle/input/metrics/ef_results_from_masks.csv")

# Keep only the first 5 rows for quick testing
df_sample = df.head(5)
print(f"🧾 Taking first {len(df_sample)} EF records for report generation...\n")

# ==========================================================
# Load lightweight summarization model (you can swap with Mistral)
# ==========================================================
model_id = "mistralai/Mistral-7B-Instruct-v0.2"
pipe = pipeline("text-generation", model=model_id, device_map="auto")

# ==========================================================
# Generate clinical summaries
# ==========================================================
for i, row in df_sample.iterrows():
    ef = row["EF_%"]
    vid = row["Video_ID"]
    prompt = (
        f"Patient {vid} has an ejection fraction (EF) of {ef:.2f}%. "
        f"Provide a concise clinical interpretation of this EF value "
        f"in the context of cardiac function."
    )
    res = pipe(prompt, max_new_tokens=80, do_sample=False)
    report = res[0]["generated_text"].split("\n")[-1].strip()

    print(f"--- Report for {vid} ---")
    print(report)
    print()
